In [ ]:
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118


In [1]:
!git clone https://github.com/Arsbul-hub/yolo8eyes_detection.git


Cloning into 'yolo8eyes_detection'...
remote: Enumerating objects: 3739, done.
remote: Counting objects: 100% (1/1), done.
remote: Total 3739 (delta 0), reused 0 (delta 0), pack-reused 3738 (from 2)
Receiving objects: 100% (3739/3739), 154.23 MiB | 17.71 MiB/s, done.
Resolving deltas: 100% (54/54), done.
Updating files: 100% (7924/7924), done.


In [12]:
%cd yolo8eyes_detection

[WinError 2] Не удается найти указанный файл: 'yolo8eyes_detection'
C:\Users\Arsbul\Desktop\yolo8eyes_detection


C:\Users\Arsbul\Desktop\yolo8eyes_detection\venv\lib\site-packages\IPython\core\magics\osm.py:393: UserWarning: This is now an optional IPython functionality, using bookmarks requires you to install the `pickleshare` library.
  bkms = self.shell.db.get('bookmarks', {})


In [1]:
import torch


In [2]:
from pathlib import Path
from PIL import Image
import os
from torch.utils.data import Dataset
from torchvision import transforms


class FaceDataset(Dataset):

    def __init__(self, root):

        self.samples = []

        self.classes = set()
        for i in os.listdir(root):
            fn = i.split("_")[0]
            self.classes.add(fn)
        self.classes = list(self.classes)
        
        
        self.class_to_idx = {
            c: i
            for i, c in enumerate(self.classes)
        }

        for i in os.listdir(root):
            person = i.split("_")[0]
            self.samples.append(
                (
                    os.path.join(root, i),
                    self.class_to_idx[person]
                )
            )
        w = 160
        k = 0.43 # примерно
        self.transform = transforms.Compose([

            transforms.Resize((int(w * k),w)),

            transforms.RandomHorizontalFlip(p=0.5),

            transforms.RandomRotation(5),
            
            transforms.ColorJitter(
                brightness=0.2,
                contrast=0.2,
                saturation=0.15,
                hue=0.02
            ),

            transforms.ToTensor(),

            transforms.Normalize(
                mean=[0.485, 0.456, 0.406],
                std=[0.229, 0.224, 0.225]
            )

        ])

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):

        path, label = self.samples[idx]

        image = Image.open(path).convert("RGB")

        image = self.transform(image)

        return image, label

In [3]:
from torch.utils.data import Dataset, DataLoader, random_split
dataset = FaceDataset("eyes_train_resnet")

val_size = int(0.2 * len(dataset))
train_size = len(dataset) - val_size

train_dataset, val_dataset = random_split(
    dataset, 
    [train_size, val_size],
    generator=torch.Generator().manual_seed(42)  # Для воспроизводимости
)

In [4]:
print(dataset.class_to_idx)
print(os.listdir("eyes_train_resnet")[0])

{'Kashyap': 0, 'Amitabh Bachchan': 1, 'Robert Downey Jr': 2, 'Andy Samberg': 3, 'Dwayne Johnson': 4, 'Tom Cruise': 5, 'Alia Bhatt': 6, 'Margot Robbie': 7, 'Alexandra Daddario': 8, 'Marmik': 9, 'Jessica Alba': 10, 'Natalie Portman': 11, 'Billie Eilish': 12, 'Anushka Sharma': 13, 'Charlize Theron': 14, 'Lisa Kudrow': 15, 'Vijay Deverakonda': 16, 'Courtney Cox': 17, 'Brad Pitt': 18, 'Claire Holt': 19, 'Henry Cavill': 20, 'Priyanka Chopra': 21, 'Hrithik Roshan': 22, 'Elizabeth Olsen': 23, 'Camila Cabello': 24, 'Zac Efron': 25, 'Hugh Jackman': 26, 'Virat Kohli': 27, 'Akshay Kumar': 28, 'Ellen Degeneres': 29, 'Roger Federer': 30}
Akshay Kumar_10.jpg


In [5]:
from torch.utils.data import DataLoader

loader = DataLoader(

    train_dataset,

    batch_size=64,

    shuffle=True,

    num_workers=0

)
loaderv = DataLoader(

    val_dataset,

    batch_size=64,

    shuffle=False,

    num_workers=0

)

In [6]:
import torch.nn as nn

from torchvision.models import resnet18, ResNet18_Weights


class FaceNet(nn.Module):

    def __init__(self):

        super().__init__()

        self.backbone = resnet18(weights=ResNet18_Weights.DEFAULT)

        self.backbone.fc = nn.Linear(512,512)

    def forward(self,x):

        x = self.backbone(x)

        return x

In [7]:
device = "cuda"
model = FaceNet().to(device)

In [36]:
!pip install pytorch-metric-learning

In [8]:
from pytorch_metric_learning.losses import ArcFaceLoss

In [9]:

epochs = 100

In [10]:
loss_fn = ArcFaceLoss(

    num_classes=len(dataset.classes),

    embedding_size=512

)

In [11]:
optimizer = torch.optim.AdamW(
    list(model.parameters()) + list(loss_fn.parameters()),
    lr=1e-3, weight_decay=5e-4
)

In [12]:
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=epochs  # число эпох
)

In [13]:
no_train_cols = 0
min_loss = None
for epoch in range(epochs):
    model.train()
    tloss = 0.0
    val_loss = 0.0

    for images, labels in loader:

        images = images.to(device)

        labels = labels.to(device)

        embeddings = model(images)

        loss = loss_fn(
            embeddings,
            labels
        )
        tloss += loss.item()
        optimizer.zero_grad()

        loss.backward()

        optimizer.step()
    scheduler.step()

    model.eval()
    with torch.no_grad():
        for images, labels in loaderv:
            images, labels = images.to(device), labels.to(device)
            embeddings = model(images)
            loss = loss_fn(embeddings, labels)
            val_loss += loss.item()
        
    if min_loss is None or val_loss / len(loaderv) < min_loss:
        torch.save(model.state_dict(), "resnet_eyes_best.pth")
        min_loss = val_loss / len(loaderv)
        no_train_cols = 0
    else:
        no_train_cols += 1
    if no_train_cols > 20:
        print("Модель не обучается последние 20 эпох")
    torch.save(model.state_dict(), "resnet_eyes_last.pth")
    print(f"TLoss: {tloss / len(loader)}, ValLoss: {val_loss / len(loaderv)}")

TLoss: 34.22110961914063, ValLoss: 34.70124053955078
TLoss: 27.926987075805663, ValLoss: 27.572611945016043
TLoss: 23.503906478881834, ValLoss: 36.530178615025115
TLoss: 19.097336769104004, ValLoss: 29.598411560058594
TLoss: 16.037199249267577, ValLoss: 27.573629106794083
TLoss: 10.867819023132324, ValLoss: 22.118337631225586
TLoss: 9.452403469085693, ValLoss: 17.392272812979563
TLoss: 7.821430234909058, ValLoss: 19.702621459960938
TLoss: 6.703089570999145, ValLoss: 18.31343514578683
TLoss: 5.088126716613769, ValLoss: 13.41122041429792
TLoss: 4.744593544006348, ValLoss: 14.286792073931013
TLoss: 5.015917315483093, ValLoss: 15.656404086521693
TLoss: 4.504486570358276, ValLoss: 13.822894096374512
TLoss: 3.848760948181152, ValLoss: 8.420679136767701
TLoss: 3.032799367904663, ValLoss: 11.39860132762364
TLoss: 3.6515476083755494, ValLoss: 15.093931879316058
TLoss: 2.598266669511795, ValLoss: 16.421468644560914
TLoss: 2.4603702926635744, ValLoss: 12.038268498011998
TLoss: 2.661114113330841, 

In [25]:
model = FaceNet()
model.load_state_dict(torch.load("resnet_eyes_last.pth"))
model = model.to(device)

ERROR: Could not install packages due to an OSError: [WinError 5] Отказано в доступе: 'C:\\Users\\Arsbul\\Desktop\\yolo8eyes_detection\\venv\\Lib\\site-packages\\~orch\\lib\\asmjit.dll'
Check the permissions.




  Attempting uninstall: torch
    Found existing installation: torch 2.7.1+cu118
    Uninstalling torch-2.7.1+cu118:
      Successfully uninstalled torch-2.7.1+cu118


In [43]:
for i1, (img1, l1) in enumerate(dataset):
  for i2, (img2, l2) in enumerate(dataset):

    emb1 = model(img1.unsqueeze(0).to(device))
    emb2 = model(img2.unsqueeze(0).to(device))
    similarity = torch.nn.functional.cosine_similarity(
        emb1,
        emb2
    )
    print(f"Для {l1} и {l2} (инд {i2}) схожесть {similarity}")


Для 29 и 29 (инд 0) схожесть tensor([0.9368], device='cuda:0', grad_fn=<SumBackward1>)
Для 29 и 29 (инд 1) схожесть tensor([0.9814], device='cuda:0', grad_fn=<SumBackward1>)
Для 29 и 29 (инд 2) схожесть tensor([0.9889], device='cuda:0', grad_fn=<SumBackward1>)
Для 29 и 29 (инд 3) схожесть tensor([0.9728], device='cuda:0', grad_fn=<SumBackward1>)
Для 29 и 29 (инд 4) схожесть tensor([0.8621], device='cuda:0', grad_fn=<SumBackward1>)
Для 29 и 29 (инд 5) схожесть tensor([0.9932], device='cuda:0', grad_fn=<SumBackward1>)
Для 29 и 29 (инд 6) схожесть tensor([0.9877], device='cuda:0', grad_fn=<SumBackward1>)
Для 29 и 29 (инд 7) схожесть tensor([0.9796], device='cuda:0', grad_fn=<SumBackward1>)
Для 29 и 29 (инд 8) схожесть tensor([0.9895], device='cuda:0', grad_fn=<SumBackward1>)
Для 29 и 29 (инд 9) схожесть tensor([0.9641], device='cuda:0', grad_fn=<SumBackward1>)
Для 29 и 29 (инд 10) схожесть tensor([0.9648], device='cuda:0', grad_fn=<SumBackward1>)
Для 29 и 29 (инд 11) схожесть tensor([0.98

KeyboardInterrupt: 

In [120]:
model.eval()
img1, l1 = dataset[0]
img2, l2 = None, None
for n, (i, l) in enumerate(dataset):
  if l != l1:
    img2, l2 = i, l
    break
img1, l1 = dataset[1]
img2, l2 = dataset[2]
print(dataset.samples[0], dataset.samples[n], dataset[n])

emb1 = model(img1.unsqueeze(0).to(device))
emb2 = model(img2.unsqueeze(0).to(device))
similarity = torch.nn.functional.cosine_similarity(
    emb1,
    emb2
)
print(f"Для {l1} и {l2} схожесть {similarity}")


(PosixPath('resnet_train_aligned/Akshay Kumar/Akshay Kumar_11.jpg'), 0) (PosixPath('resnet_train_aligned/Alexandra Daddario/Alexandra Daddario_70.jpg'), 1) (tensor([[[-0.8667, -0.8667, -0.8667,  ..., -0.7176, -0.7098, -0.7098],
         [-0.8667, -0.8667, -0.8667,  ..., -0.7333, -0.7333, -0.7333],
         [-0.8667, -0.8667, -0.8667,  ..., -0.7569, -0.7569, -0.7569],
         ...,
         [ 0.7412,  0.7490,  0.7804,  ..., -0.6549, -0.6078, -0.5922],
         [ 0.7333,  0.7490,  0.7725,  ..., -0.6471, -0.6078, -0.5922],
         [ 0.7255,  0.7412,  0.7725,  ..., -0.6471, -0.6000, -0.5843]],

        [[-0.9373, -0.9373, -0.9373,  ..., -0.8745, -0.8667, -0.8667],
         [-0.9373, -0.9373, -0.9373,  ..., -0.8902, -0.8902, -0.8902],
         [-0.9373, -0.9373, -0.9373,  ..., -0.9137, -0.9137, -0.9137],
         ...,
         [ 0.7647,  0.7725,  0.7804,  ..., -0.8039, -0.7725, -0.7569],
         [ 0.7569,  0.7725,  0.7725,  ..., -0.7961, -0.7725, -0.7569],
         [ 0.7490,  0.7647,  0.7

In [22]:
model.eval()

FaceNet(
  (backbone): ResNet(
    (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
    (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (relu): ReLU(inplace=True)
    (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
    (layer1): Sequential(
      (0): BasicBlock(
        (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (relu): ReLU(inplace=True)
        (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      )
      (1): BasicBlock(
        (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_ru

In [26]:
w = 160
k = 0.43 # примерно
transform = transforms.Compose([

    transforms.Resize((int(w * k),w)),

    transforms.ToTensor(),

    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )

])

ref = transform(Image.open("./test_eyes.jpg").convert("RGB"))
an = transform(Image.open("./another_eyes.jpg").convert("RGB"))
test = transform(Image.open("./WIN_20260704_20_00_16_Pro.jpg").convert("RGB"))
emb1 = model(ref.unsqueeze(0).to(device))
emb2 = model(an.unsqueeze(0).to(device))
emb3 = model(test.unsqueeze(0).to(device))
similarity1 = torch.nn.functional.cosine_similarity(
    emb1,emb3

)
similarity2 = torch.nn.functional.cosine_similarity(
    emb1,
    emb2
)
similarity3 = torch.nn.functional.cosine_similarity(
    emb2,
    emb3
)
print(f"sim: {similarity1}")
print(f"an: {similarity2}")
print(f"an: {similarity3}")


sim: tensor([-0.1989], device='cuda:0', grad_fn=<SumBackward1>)
an: tensor([0.2123], device='cuda:0', grad_fn=<SumBackward1>)
an: tensor([0.3513], device='cuda:0', grad_fn=<SumBackward1>)


In [50]:
photos = {"ars": transform(Image.open("./WIN_20260704_20_00_16_Pro.jpg").convert("RGB")),
            "ilush": transform(Image.open("./another_eyes.jpg").convert("RGB"))}
embs = {}
for key, img in photos.items():
    embs[key] = model(img.unsqueeze(0).to(device))


In [51]:
import cv2
import onnxruntime
import numpy as np
session = onnxruntime.InferenceSession(
    "YOLOX/eye_detector_tiny.onnx",
    providers=["CUDAExecutionProvider"]
)
from yolox.data.data_augment import preproc as preprocess
# from yolox.data.datasets import COCO_CLASSES
from yolox.utils import mkdir, multiclass_nms, demo_postprocess, vis

CLASSES = ["eyes"]

input_shape = (640, 640)
cap = cv2.VideoCapture(0)

while True:
    ret, frame = cap.read()
    img, ratio = preprocess(frame, input_shape)

    
    ort_inputs = {session.get_inputs()[0].name: img[None, :, :, :]}
    output = session.run(None, ort_inputs)
    predictions = demo_postprocess(output[0], input_shape)[0]

    boxes = predictions[:, :4]

    scores = predictions[:, 4:5] * predictions[:, 5:]

    boxes_xyxy = np.ones_like(boxes)
    boxes_xyxy[:, 0] = boxes[:, 0] - boxes[:, 2]/2.
    boxes_xyxy[:, 1] = boxes[:, 1] - boxes[:, 3]/2.
    boxes_xyxy[:, 2] = boxes[:, 0] + boxes[:, 2]/2.
    boxes_xyxy[:, 3] = boxes[:, 1] + boxes[:, 3]/2.
    boxes_xyxy /= ratio
    dets = multiclass_nms(boxes_xyxy, scores, nms_thr=0.45, score_thr=0.1)
    if dets is not None:
        final_boxes, final_scores, final_cls_inds = dets[:, :4], dets[:, 4], dets[:, 5]
        frame = vis(frame, final_boxes, final_scores, final_cls_inds,
                         conf=0.3, class_names=CLASSES)
        max_bs = max(zip(final_boxes, final_scores), key=lambda a:a[1])  
        box, score = max_bs[0], max_bs[1]
        
        frame_cropped = frame[round(box[1].item()):round(box[3].item()), round(box[0].item()):round(box[2].item())]
        embg = model(transform(Image.fromarray(frame_cropped).convert("RGB")).unsqueeze(0).to(device))
        score = None
        mk = None
        print(f"ars: {torch.nn.functional.cosine_similarity(embs['ars'], embg).item()}, ilush: {torch.nn.functional.cosine_similarity(embs['ilush'], embg).item()}")
        for key, emb in embs.items():
            sc = torch.nn.functional.cosine_similarity(emb, embg).item()
            
            if score is None or sc > score:
                score = sc
                
                mk = key
        score = np.clip(score, 0, 1).item()
        
        frame = cv2.putText(frame, f"{mk}: {int(score * 100)}%", (40, 40), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, int(255 * score), int(255 * (1 - score))), 2)
        
    cv2.imshow("Camera", frame)

    if cv2.waitKey(1) == 27:
        break
cap.release()
cv2.destroyAllWindows()


ars: 0.24817103147506714, ilush: 0.5212698578834534
ars: 0.04825364053249359, ilush: 0.6521531343460083
ars: 0.305235892534256, ilush: 0.7489369511604309
ars: 0.17220532894134521, ilush: 0.7308457493782043
ars: 0.10784079134464264, ilush: 0.7111363410949707
ars: 0.1478154957294464, ilush: 0.698995053768158
ars: 0.030891362577676773, ilush: 0.5368510484695435
ars: 0.12072600424289703, ilush: 0.7618865966796875
ars: 0.14954626560211182, ilush: 0.7881847023963928
ars: 0.15006348490715027, ilush: 0.7755231857299805
ars: 0.1544073522090912, ilush: 0.6846151351928711
ars: 0.20071496069431305, ilush: 0.8096417784690857
ars: 0.22766482830047607, ilush: 0.8436503410339355
ars: 0.21804943680763245, ilush: 0.7513574361801147
ars: 0.2073148787021637, ilush: 0.7738081216812134
ars: 0.23814821243286133, ilush: 0.7656083106994629
ars: 0.18929561972618103, ilush: 0.7904365062713623
ars: 0.1934225857257843, ilush: 0.7519034147262573
ars: 0.14746296405792236, ilush: 0.6516724228858948
ars: 0.26589745283